In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import re
from cycler import cycler

# ==========================================
# CONFIGURATION
# ==========================================

servicecolors = ["#115e67", "#f0b434", "#ff5733", "#800000"]
plt.rcParams['axes.prop_cycle'] = cycler(color=servicecolors)

ROOT_DIR = "./RESULTADOS_TESTBED_3GPP/RESULTADOS_FINALES_CMP_sched_att_0to18dB_3gpp/results_CAP20"

schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

SWEEP_DIR_PATTERN = "GAMMA_CRIT_*"
ITERS = "ITER*"

# Log configuration
IDX_TIME = 0
IDX_SEQ = 1
IDX_DATA = 1

HAS_HEADER = False

# Timestamp conversion: microseconds -> milliseconds
TIME_MULT_FACTOR = 1e-3

# Packet size (bytes), used only if the logs do not contain packet sizes
PACKET_SIZE_BYTES = 1000

STREAMS_INFO = {
    "1_1": "Stream 1",
    "1_2": "Stream 2"
}


# ==========================================
# AUXILIARY FUNCTIONS
# ==========================================

def read_log(filepath):
    """Read a whitespace-separated log."""

    if not os.path.exists(filepath):
        return None

    try:
        df = pd.read_csv(
            filepath,
            sep=r'\s+',
            header=None if not HAS_HEADER else 0,
            engine='python'
        )

        if df.empty:
            return None

        return df

    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None


def get_iteration_folders(gamma_folder):
    """Return all iteration folders."""

    return sorted(
        folder for folder in glob.glob(os.path.join(gamma_folder, ITERS))
        if os.path.isdir(folder)
    )


# ==========================================
# THROUGHPUT CALCULATION
# ==========================================

def calculate_throughput_mbps(gamma_folder, suffix_stream):

    all_bitrates = []

    iter_folders = get_iteration_folders(gamma_folder)

    for iter_folder in iter_folders:

        f_srv = os.path.join(
            iter_folder,
            "go_server",
            f"logServer_{suffix_stream}.log"
        )

        df = read_log(f_srv)

        if df is None:
            continue

        timestamps = df.iloc[:, IDX_TIME].to_numpy()

        if len(timestamps) < 2:
            continue

        duration_s = (timestamps.max() - timestamps.min()) / 1e6

        if duration_s <= 0:
            continue

        # Assumption: fixed packet size
        total_bits = len(df) * PACKET_SIZE_BYTES * 8

        throughput_mbps = total_bits / (duration_s * 1e6)

        all_bitrates.append(throughput_mbps)

    if not all_bitrates:
        return np.nan

    return np.mean(all_bitrates)


# ==========================================
# DELAY CALCULATION
# ==========================================

def process_stream_logs(gamma_folder, suffix_stream):

    all_delays = []

    iter_folders = get_iteration_folders(gamma_folder)

    if not iter_folders:
        print(f"No iterations found in {gamma_folder}")
        return np.array([]), np.nan

    for iter_folder in iter_folders:

        f_cli = os.path.join(
            iter_folder,
            "go_client",
            f"logClient_{suffix_stream}.log"
        )

        f_srv = os.path.join(
            iter_folder,
            "go_server",
            f"logServer_{suffix_stream}.log"
        )

        df_tx = read_log(f_cli)
        df_rx = read_log(f_srv)

        if df_tx is None or df_rx is None:
            print(f"Missing or empty logs in {iter_folder}, stream {suffix_stream}")
            continue

        # Extract timestamps and packet sequence numbers
        tx = df_tx.iloc[:, [IDX_SEQ, IDX_TIME]].copy()
        rx = df_rx.iloc[:, [IDX_SEQ, IDX_TIME]].copy()

        tx.columns = ["seq", "tx_time"]
        rx.columns = ["seq", "rx_time"]

        # Convert sequence numbers to a common type
        tx["seq"] = pd.to_numeric(tx["seq"], errors="coerce")
        rx["seq"] = pd.to_numeric(rx["seq"], errors="coerce")

        tx["tx_time"] = pd.to_numeric(tx["tx_time"], errors="coerce")
        rx["rx_time"] = pd.to_numeric(rx["rx_time"], errors="coerce")

        tx = tx.dropna()
        rx = rx.dropna()

        # Match packets using sequence numbers
        matched = pd.merge(
            tx,
            rx,
            on="seq",
            how="inner",
            validate="one_to_one"
        )

        if matched.empty:
            print(f"No matching packets in {iter_folder}, stream {suffix_stream}")
            continue

        # Calculate E2E delay
        delays_ms = (
            matched["rx_time"].to_numpy() -
            matched["tx_time"].to_numpy()
        ) * TIME_MULT_FACTOR

        all_delays.append(delays_ms)

    if not all_delays:
        return np.array([]), np.nan

    delays_concat = np.concatenate(all_delays)

    throughput = calculate_throughput_mbps(
        gamma_folder,
        suffix_stream
    )

    return delays_concat, throughput


# ==========================================
# PER CALCULATION
# ==========================================

def process_per_logs(gamma_folder):

    all_per_values = []

    iter_folders = get_iteration_folders(gamma_folder)

    if not iter_folders:
        print(f"No iterations found in {gamma_folder}")
        return np.nan

    for iter_folder in iter_folders:

        f_per = os.path.join(
            iter_folder,
            "go_client",
            "resultado_final.log"
        )

        df_per = read_log(f_per)

        if df_per is None:
            continue

        try:
            per_value = pd.to_numeric(
                df_per.iloc[0, 2],
                errors="coerce"
            )

            if pd.notna(per_value):
                all_per_values.append(per_value)

        except (IndexError, ValueError) as e:
            print(f"Invalid PER in {f_per}: {e}")

    if not all_per_values:
        return np.nan

    return np.mean(all_per_values)


In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
from scipy import stats


# ============================================================
# CONFIGURATION
# ============================================================

ROOT_DIR = "./RESULTADOS_TESTBED_DSRC/results_SAT_MLWDF_RR_STBP/results"

SCHEDULERS = ["MLWDF", "RR", "STBP"]
MLWDF_CONFIGS = ["W1_Q1", "W1_Q0", "W0_Q1"]
ITER_PATTERN = "ITER*"

STREAMS = ["1_1", "1_2"]

IDX_TIME = 0
TIME_MULT_FACTOR = 1e-3

PACKET_SIZE_BYTES = 1000

# The original throughput calculation assumes timestamps are in
# microseconds, so (max_timestamp - min_timestamp) / 1e6 is seconds.
THROUGHPUT_TIME_DIVISOR = 1e6

CONFIDENCE = 0.95


# ============================================================
# CONFIDENCE INTERVAL
# ============================================================

def confidence_interval_from_runs(values, confidence=0.95):
    """
    Student-t confidence interval based on independent run-level values.

    Each element in 'values' must be one result per independent run:
    e.g. one mean delay, one throughput, or one packet-loss rate per run.
    """

    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    n = len(values)

    result = {
        "n_runs": n,
        "mean": np.nan,
        "std_between_runs": np.nan,
        "ci_margin": np.nan,
        "ci_low": np.nan,
        "ci_high": np.nan
    }

    if n == 0:
        return result

    mean_value = np.mean(values)
    result["mean"] = mean_value

    if n < 2:
        return result

    std_value = np.std(values, ddof=1)
    standard_error = std_value / np.sqrt(n)

    alpha = 1.0 - confidence
    t_critical = stats.t.ppf(
        1.0 - alpha / 2.0,
        df=n - 1
    )

    margin = t_critical * standard_error

    result.update({
        "std_between_runs": std_value,
        "ci_margin": margin,
        "ci_low": mean_value - margin,
        "ci_high": mean_value + margin
    })

    return result


# ============================================================
# READ ONE ITERATION
# ============================================================

def read_timestamp_column(filepath):
    """Read timestamp column from a whitespace-separated log."""

    df = pd.read_csv(
        filepath,
        sep=r"\s+",
        header=None,
        engine="python"
    )

    if df.empty or df.shape[1] <= IDX_TIME:
        return np.array([], dtype=float)

    return pd.to_numeric(
        df.iloc[:, IDX_TIME],
        errors="coerce"
    ).dropna().to_numpy(dtype=float)


def calculate_run_delay_ms(iter_folder, stream_id):
    """
    Calculate packet delays for one iteration.

    TX and RX are paired by line position. To accommodate different
    log lengths, only the first min(len(TX), len(RX)) entries are used.

    This assumes that the entries at the same position correspond
    to the same packet. If packets can be lost in the middle, matching
    by packet sequence number/ID is required instead.
    """

    tx_file = os.path.join(
        iter_folder, "go_client", f"logClient_{stream_id}.log"
    )
    rx_file = os.path.join(
        iter_folder, "go_server", f"logServer_{stream_id}.log"
    )

    if not os.path.exists(tx_file) or not os.path.exists(rx_file):
        return np.array([])

    tx_times = read_timestamp_column(tx_file)
    rx_times = read_timestamp_column(rx_file)

    n_packets = min(len(tx_times), len(rx_times))

    if n_packets == 0:
        return np.array([])

    tx_times = tx_times[:n_packets]
    rx_times = rx_times[:n_packets]

    delays_ms = (rx_times - tx_times) * TIME_MULT_FACTOR

    # Discard invalid and negative delays.
    delays_ms = delays_ms[
        np.isfinite(delays_ms) & (delays_ms >= 0)
    ]

    return delays_ms


def calculate_run_throughput_mbps(iter_folder, stream_id):
    """
    Calculate throughput for one iteration.

    Preserves the original assumptions:
    - each server-log line represents one received packet;
    - each packet has PACKET_SIZE_BYTES bytes;
    - timestamp range divided by THROUGHPUT_TIME_DIVISOR gives seconds.
    """

    rx_file = os.path.join(
        iter_folder, "go_server", f"logServer_{stream_id}.log"
    )

    if not os.path.exists(rx_file):
        return np.nan

    try:
        rx_times = read_timestamp_column(rx_file)

        if len(rx_times) < 2:
            return np.nan

        duration_s = (
            np.max(rx_times) - np.min(rx_times)
        ) / THROUGHPUT_TIME_DIVISOR

        if duration_s <= 0:
            return np.nan

        total_bits = len(rx_times) * PACKET_SIZE_BYTES * 8
        throughput_mbps = total_bits / (duration_s * 1e6)

        return throughput_mbps

    except Exception as error:
        print(f"Throughput error in {rx_file}: {error}")
        return np.nan


def calculate_run_per(iter_folder):
    """
    Read the PER value from resultado_final.log.

    Preserves the original parser: first row, third column.
    Returns NaN if the file/value cannot be read.
    """

    per_file = os.path.join(
        iter_folder, "go_client", "resultado_final.log"
    )

    if not os.path.exists(per_file):
        return np.nan

    try:
        df = pd.read_csv(
            per_file,
            header=None,
            sep=r"\s+",
            engine="python"
        )

        if df.empty or df.shape[1] < 3:
            return np.nan

        return float(df.iloc[0, 2])

    except Exception as error:
        print(f"PER error in {per_file}: {error}")
        return np.nan


# ============================================================
# PROCESS ONE SCENARIO / GAMMA FOLDER
# ============================================================

def process_scenario(folder, scheduler, config, gamma):
    """
    Computes one metric value per independent ITER for each stream,
    then calculates the 95% CI across those run-level values.

    Outputs:
      - one row per stream with delay and throughput statistics;
      - one scenario-level row with PER statistics.
    """

    iter_folders = sorted(
        glob.glob(os.path.join(folder, ITER_PATTERN))
    )

    if not iter_folders:
        print(f"No ITER folders found in: {folder}")
        return [], []

    stream_rows = []
    per_values = []

    for stream_id in STREAMS:
        run_mean_delays = []
        run_p95_delays = []
        run_throughputs = []

        for iter_folder in iter_folders:
            delays = calculate_run_delay_ms(
                iter_folder, stream_id
            )

            if len(delays) > 0:
                # One mean delay per independent execution.
                run_mean_delays.append(np.mean(delays))

                # One p95 delay per independent execution.
                run_p95_delays.append(
                    np.percentile(delays, 95)
                )

            throughput = calculate_run_throughput_mbps(
                iter_folder, stream_id
            )

            if np.isfinite(throughput):
                run_throughputs.append(throughput)

        delay_ci = confidence_interval_from_runs(
            run_mean_delays, CONFIDENCE
        )

        p95_ci = confidence_interval_from_runs(
            run_p95_delays, CONFIDENCE
        )

        throughput_ci = confidence_interval_from_runs(
            run_throughputs, CONFIDENCE
        )

        stream_rows.append({
            "scheduler": scheduler,
            "config": config,
            "gamma": gamma,
            "stream": stream_id,

            # Mean delay: CI across independent run means.
            "delay_n_runs": delay_ci["n_runs"],
            "delay_mean_ms": delay_ci["mean"],
            "delay_std_between_runs_ms":
                delay_ci["std_between_runs"],
            "delay_ci95_margin_ms": delay_ci["ci_margin"],
            "delay_ci95_low_ms": delay_ci["ci_low"],
            "delay_ci95_high_ms": delay_ci["ci_high"],

            # p95 delay: calculate p95 per run, then CI across runs.
            "p95_n_runs": p95_ci["n_runs"],
            "p95_mean_ms": p95_ci["mean"],
            "p95_std_between_runs_ms":
                p95_ci["std_between_runs"],
            "p95_ci95_margin_ms": p95_ci["ci_margin"],
            "p95_ci95_low_ms": p95_ci["ci_low"],
            "p95_ci95_high_ms": p95_ci["ci_high"],

            # Throughput: CI across per-run throughput values.
            "throughput_n_runs": throughput_ci["n_runs"],
            "throughput_mean_mbps": throughput_ci["mean"],
            "throughput_std_between_runs_mbps":
                throughput_ci["std_between_runs"],
            "throughput_ci95_margin_mbps":
                throughput_ci["ci_margin"],
            "throughput_ci95_low_mbps":
                throughput_ci["ci_low"],
            "throughput_ci95_high_mbps":
                throughput_ci["ci_high"]
        })

    # PER is scenario-level in the original parser, not stream-specific.
    for iter_folder in iter_folders:
        per_value = calculate_run_per(iter_folder)

        if np.isfinite(per_value):
            per_values.append(per_value)

    per_ci = confidence_interval_from_runs(
        per_values, CONFIDENCE
    )

    per_rows = [{
        "scheduler": scheduler,
        "config": config,
        "gamma": gamma,
        "per_n_runs": per_ci["n_runs"],
        "per_mean": per_ci["mean"],
        "per_std_between_runs": per_ci["std_between_runs"],
        "per_ci95_margin": per_ci["ci_margin"],
        "per_ci95_low": per_ci["ci_low"],
        "per_ci95_high": per_ci["ci_high"]
    }]

    return stream_rows, per_rows


# ============================================================
# MAIN
# ============================================================

all_stream_results = []
all_per_results = []

for scheduler in SCHEDULERS:

    configs = (
        MLWDF_CONFIGS
        if scheduler == "MLWDF"
        else [""]
    )

    for config in configs:

        base_dir = os.path.join(
            ROOT_DIR,
            scheduler,
            config
        )

        if not os.path.isdir(base_dir):
            print(f"Directory not found: {base_dir}")
            continue

        if scheduler == "MLWDF":
            gamma_folders = sorted(
                glob.glob(
                    os.path.join(base_dir, "GAMMA_CRIT_*")
                )
            )

            if not gamma_folders:
                gamma_folders = [base_dir]
        else:
            gamma_folders = [base_dir]

        for folder in gamma_folders:

            match = re.search(
                r"GAMMA_CRIT_([\d.eE+-]+)",
                os.path.basename(folder)
            )

            gamma = (
                float(match.group(1))
                if match
                else np.nan
            )

            stream_rows, per_rows = process_scenario(
                folder=folder,
                scheduler=scheduler,
                config=config,
                gamma=gamma
            )

            all_stream_results.extend(stream_rows)
            all_per_results.extend(per_rows)


# ============================================================
# EXPORT RESULTS
# ============================================================

if all_stream_results:
    df_streams = pd.DataFrame(all_stream_results)

    df_streams = df_streams.sort_values(
        ["scheduler", "config", "stream", "gamma"],
        na_position="first"
    )

    df_streams.to_csv(
        "statistical_uncertainty_streams.csv",
        index=False,
        float_format="%.6f"
    )

    print(
        "\nSaved: statistical_uncertainty_streams.csv"
    )
else:
    print("\nNo stream results were generated.")


if all_per_results:
    df_per = pd.DataFrame(all_per_results)

    df_per = df_per.sort_values(
        ["scheduler", "config", "gamma"],
        na_position="first"
    )

    df_per.to_csv(
        "statistical_uncertainty_per.csv",
        index=False,
        float_format="%.6f"
    )

    print("Saved: statistical_uncertainty_per.csv")
else:
    print("No PER results were generated.")


print("\nProcessing finished.")


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

# ==========================================
# CONFIGURACIÓN
# ==========================================
rOOT_DIR = "./RESULTADOS_TESTBED_3GPP/RESULTADOS_FINALES_CMP_sched_att_0to18dB_3gpp/results_CAP30"
schednames = ["MLWDF"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

def calculate_throughput_mbps(gamma_folder, suffix_stream):
    """
    Calcula la tasa binaria promedio (Mbps) para un stream sumando 
    los paquetes recibidos en el servidor.
    """
    all_bitrates = []
    iter_folders = sorted(glob.glob(os.path.join(gamma_folder, "ITER*")))
    
    for iter_folder in iter_folders:
        f_srv = os.path.join(iter_folder, "go_server", f"logServer_{suffix_stream}.log")
        
        if not os.path.exists(f_srv):
            continue
            
        try:
            df = pd.read_csv(f_srv, sep=r'\s+', header=None, engine='python')
            if df.empty: continue
            
            duration = (df.iloc[:, 0].max() - df.iloc[:, 0].min()) / 1e6
            if duration <= 0: continue
            
            total_bits = len(df) * 1000 * 8
            mbps = total_bits / (duration * 1e6)
            # print(f"Iteración: {iter_folder}, Stream: {suffix_stream}, Duration: {duration:.2f}s, Total bits: {total_bits}, Mbps: {mbps:.2f}")
            all_bitrates.append(mbps)
        except:
            continue
            
    return np.mean(all_bitrates) if all_bitrates else np.nan

def plot_instantaneous_throughput(log_file, window_size=1.0):
    """
    Calcula y grafica el throughput instantáneo de un único log.
    """
    try:
        df = pd.read_csv(log_file, sep=r'\s+', header=None, engine='python')
        if df.empty:
            print(f"El log {log_file} está vacío.")
            return

        # Convertir microsegundos a segundos relativos
        timestamps_sec = (df.iloc[:, 0] - df.iloc[:, 0].min()) / 1e6
        
        # Asignar a ventanas de tiempo
        time_bins = (timestamps_sec // window_size) * window_size
        
        # Calcular Mbps por ventana
        bits_per_packet = 1000 * 8
        instant_mbps = df.groupby(time_bins).size() * bits_per_packet / (window_size * 1e6)
        
        # Graficar
        plt.figure(figsize=(12, 5))
        plt.plot(instant_mbps.index, instant_mbps.values, marker='.', linestyle='-', color='#115e67', alpha=0.8)
        plt.title(f'Throughput Instantáneo (Ventana: {window_size}s)\nArchivo: {os.path.basename(log_file)}', fontsize=12)
        plt.xlabel('Tiempo (segundos)', fontsize=12)
        plt.ylabel('Throughput (Mbps)', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error procesando {log_file}: {e}")

# ==========================================
# PROCESAMIENTO PROMEDIO TOTAL
# ==========================================
throughput_results = []
ejemplo_log_instantaneo = True  # Variable para guardar un log de prueba

for schedname in schednames:
    for config in subschedconfigs:
        if schedname == "MLWDF":
            BASE_DIR = os.path.join(rOOT_DIR, schedname, config)
            pattern = os.path.join(BASE_DIR, "GAMMA_CRIT_*")
        else:
            BASE_DIR = os.path.join(rOOT_DIR, schedname)
            pattern = os.path.join(BASE_DIR, "*")

        folders = sorted(glob.glob(pattern))
        
        for folder in folders:
            if not os.path.isdir(folder): continue
            
            # Capturar el primer log válido que encuentre para usarlo de ejemplo luego
            if ejemplo_log_instantaneo is None:
                test_log = os.path.join(folder, "ITER_1", "go_client", "logClient_1_1.log")
                if os.path.exists(test_log):
                    ejemplo_log_instantaneo = test_log

            gamma_val = 1.0
            if schedname == "MLWDF":
                match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
                gamma_val = float(match.group(1)) if match else 1.0
            
            thr_1_1 = calculate_throughput_mbps(folder, "1_1")
            thr_1_2 = calculate_throughput_mbps(folder, "1_2")
            
            throughput_results.append({
                'Scheduler': schedname,
                'Config': config,
                'Gamma': gamma_val,
                'Stream 1.1 (Mbps)': thr_1_1,
                'Stream 1.2 (Mbps)': thr_1_2,
                'Total (Mbps)': thr_1_1 + thr_1_2
            })

df_thr = pd.DataFrame(throughput_results)

# ==========================================
# GRÁFICA 1: THROUGHPUT PROMEDIO
# ==========================================
if not df_thr.empty:
    plt.figure(figsize=(12, 7))
    colors = {'W1_Q1': '#115e67', 'W1_Q0': '#f0b434', 'W0_Q1': '#ff5733'}

    for config in subschedconfigs:
        df_c = df_thr[df_thr['Config'] == config]
        if df_c.empty: continue
        
        mlwdf = df_c[df_c['Scheduler'] == "MLWDF"].sort_values('Gamma')
        plt.plot(mlwdf['Gamma'], mlwdf['Total (Mbps)'], 'o-', 
                 label=f'MLWDF {config} (Total)', color=colors.get(config, 'blue'), linewidth=2)
        
        for other in ["RR", "STBP"]:
            val = df_c[df_c['Scheduler'] == other]['Total (Mbps)'].mean()
            if not np.isnan(val):
                plt.axhline(y=val, color=colors.get(config, 'blue'), linestyle='--', alpha=0.6,
                            label=f'{other} {config}')

    plt.title('Tasa Binaria Total Promedio (Throughput)', fontsize=14)
    plt.xlabel('Gamma Critical (Ratio)', fontsize=12)
    plt.ylabel('Mbps', fontsize=12)
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron datos para generar la gráfica promedio.")

# ==========================================
# GRÁFICA 2: THROUGHPUT INSTANTÁNEO (EJEMPLO)
# ==========================================
if ejemplo_log_instantaneo and os.path.exists(ejemplo_log_instantaneo):
    print(f"\nGenerando gráfica de throughput instantáneo para: {ejemplo_log_instantaneo}")
    # Puedes cambiar el window_size (ej. 0.1 para más detalle o 1.0 para más suavizado)
    plot_instantaneous_throughput(ejemplo_log_instantaneo, window_size=0.5)
else:
    print("\nNo se encontró un archivo de log de ejemplo para graficar el throughput instantáneo.")
    # Si quieres usar uno específico, descomenta la siguiente línea y pon la ruta correcta:
    # plot_instantaneous_throughput("TU_RUTA/logClient_1_1.log", window_size=0.5)

In [ ]:
plt.figure(figsize=(12, 7))
colors = {'W1_Q1': '#115e67', 'W1_Q0': '#f0b434', 'W0_Q1': '#ff5733'}

for config in subschedconfigs:
    df_c = df_thr[df_thr['Config'] == config]
    if df_c.empty: continue
    
    mlwdf = df_c[df_c['Scheduler'] == "MLWDF"].sort_values('Gamma')
    plt.plot(mlwdf['Gamma'], mlwdf['Stream 1.1 (Mbps)'], 'o-', 
                label=f'MLWDF {config} (Stream 1.1)', color=colors.get(config, 'blue'), linewidth=2)
    plt.plot(mlwdf['Gamma'], mlwdf['Stream 1.2 (Mbps)'], 'o-', 
                label=f'MLWDF {config} (Stream 1.2)', color=colors.get(config, 'blue'), linewidth=2)
    for other in ["RR", "STBP"]:
        val = df_c[df_c['Scheduler'] == other]['Total (Mbps)'].mean()
        if not np.isnan(val):
            plt.axhline(y=val, color=colors.get(config, 'blue'), linestyle='--', alpha=0.6,
                        label=f'{other} {config}')

plt.title('Tasa Binaria Total Promedio (Throughput)', fontsize=14)
plt.xlabel('Gamma Critical (Ratio)', fontsize=12)
plt.ylabel('Mbps', fontsize=12)
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Visualize and Saving CSVs of delay ratio

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

# ==========================================
# CONFIGURACIÓN
# ==========================================
rOOT_DIR = "./RESULTADOS_TESTBED_DSRC/results_MLWDF_STBP_RR/results"
schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

final_ratios = []

# ==========================================
# PROCESAMIENTO
# ==========================================
for schedname in schednames:
    if schedname == "MLWDF":
        for subschedconfig in subschedconfigs:
            BASE_DIR = os.path.join(rOOT_DIR, schedname, subschedconfig)
            # Definir patrón de búsqueda
            print(f"Procesando {schedname} - {subschedconfig} en {BASE_DIR}...")
            pattern = os.path.join(BASE_DIR, "GAMMA_CRIT_**")
            gamma_folders = sorted(glob.glob(pattern))
        
            print(f"Carpetas encontradas: {len(gamma_folders)}")
            for folder in gamma_folders:
                # Extraer Gamma para el eje X
                gamma_val = 0.0
                if schedname == "MLWDF":
                    match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
                    gamma_val = float(match.group(1)) if match else 0.0

                # 1. Obtener delays de ambos streams
                delays_1 = process_stream_logs(folder, "1_1")
                delays_2 = process_stream_logs(folder, "1_2")
                per_mean = process_per_logs(folder)
                # 2. Calcular el ratio de los valores esperados (promedios)
                if (isinstance(delays_1, np.ndarray) and len(delays_1) > 0 and 
                    isinstance(delays_2, np.ndarray) and len(delays_2) > 0):
                    
                    e_delay_1 = np.mean(delays_1)
                    e_delay_2 = np.mean(delays_2)
                    
                    # Evitar división por cero
                    ratio = e_delay_1 / e_delay_2 if e_delay_2 != 0 else np.nan
                    
                    final_ratios.append({
                        'Scheduler': schedname,
                        'Config': subschedconfig,
                        'gamma': gamma_val,
                        'ratio': ratio,
                        'per_mean': per_mean
                    })

    else: 
        BASE_DIR = os.path.join(rOOT_DIR, schedname)
        print(f"Procesando {schedname} en {BASE_DIR}...")
        # Procesar Stream 1_1
        delays_1_1 = process_stream_logs(BASE_DIR, "1_1")

        # Procesar Stream 1_2
        delays_1_2 = process_stream_logs(BASE_DIR, "1_2")
        if delays_1_1 is np.nan or delays_1_2 is np.nan:
            print(f"  Error leyendo logs en {folder}, saltando...")
            continue
        e_delay_1 = np.mean(delays_1_1)
        e_delay_2 = np.mean(delays_1_2)
        ratio = e_delay_1 / e_delay_2 
        print(ratio)
        # Procesar PER
        per_mean = process_per_logs(BASE_DIR)

    
        final_ratios.append({
            'Scheduler': schedname,
            'Config': "N/A",
            'gamma': "No Sweep",
            'ratio': ratio,
            'per_mean': per_mean
            })
    
        
        
   
df_plot = pd.DataFrame(final_ratios)

# ==========================================
# GRÁFICA POR CADA SUBSCHEDCONFIG
# ==========================================
plt.figure(figsize=(12, 7))

for sched in schednames:
    if sched == "MLWDF":
        for subschedconfig in subschedconfigs:
            # Filtramos y ordenamos por gamma inmediatamente
            df_c = df_plot[(df_plot['Config'] == subschedconfig) & 
                           (df_plot['Scheduler'] == "MLWDF")].sort_values('gamma')
            
            if not df_c.empty:
                plt.plot(df_c['gamma'], df_c['ratio'], 'o-', label=f'MLWDF {subschedconfig}')
                # El CSV ahora guardará los datos en orden ascendente de gamma
                df_c.to_csv(f"ratio{sched}{subschedconfig}.csv", 
                           columns=['gamma', 'ratio', 'per_mean'], index=False)
    else:
        # Filtramos y ordenamos por gamma para el resto de schedulers
        df_c = df_plot[df_plot['Scheduler'] == sched].sort_values('gamma')
        
        # Calculamos la media (el orden no afecta al promedio, pero sí al CSV)
        val = np.mean(df_c['ratio'])
        print(f"{sched} (avg ratio): {val}")
        
        if not np.isnan(val):
            plt.axhline(y=val, label=f'{sched} (avg)', linestyle='--', alpha=0.5)
            
        # El CSV ahora guardará los datos en orden ascendente de gamma
        df_c.to_csv(f"ratio{sched}.csv", columns=['gamma', 'ratio', 'per_mean'], index=False)

# Configuración estética del gráfico
plt.title(r'Ratio de Retardos Promedio: $E[Delay_{1.1}] / E[Delay_{1.2}]$')
plt.xlabel('Gamma Critical (Ratio)')
plt.ylabel('Ratio (Stream 1 / Stream 2)')
plt.grid(True, which='both', linestyle=':', alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from cycler import cycler

# ==========================================
# CONFIGURACIÓN GLOBAL DE ESTILO
# ==========================================
servicecolors = ["#115e67", "#f0b434", "#ff5733", "#800000"]
plt.rcParams['axes.prop_cycle'] = cycler(color=servicecolors)
sns.set_theme(style="whitegrid")

# ==========================================
# FUNCIONES DE APOYO
# ==========================================

def get_full_data_and_stats(folder_path, gamma_label):
    """Calcula estadísticas y devuelve DataFrame long-form para Seaborn."""
    stats_rows = []
    raw_data_list = []
    
    for stream_id in ["1_1", "1_2"]:
        # NOTA: Asegúrate de tener importada o definida tu función process_stream_logs
        delays, thput_mbps = process_stream_logs(folder_path, stream_id)
        
        if isinstance(delays, np.ndarray) and len(delays) > 0:
            d_sorted = np.sort(delays)
            lq, med, uq = np.percentile(d_sorted, [25, 50, 75])
            iqr = uq - lq
            lw = d_sorted[d_sorted >= (lq - 1.5 * iqr)].min()
            uw = d_sorted[d_sorted <= (uq + 1.5 * iqr)].max()

            # --- NUEVO: Cálculo de la tasa de fallos de deadline (paquetes > 100 ms) ---
            miss_rate_100ms = np.mean(d_sorted > 10) * 100
            if gamma_label == "No Sweep":
                gamma_label = 0
            stats_rows.append({
                'gamma': float(gamma_label), 'lw': lw, 'lq': lq, 'med': med, 
                'uq': uq, 'uw': uw, 'xaux': iqr, 'avg': d_sorted.mean(), 
                'max': d_sorted.max(), 'min': d_sorted.min(), 'std': d_sorted.std(),
                'thput_mbps': thput_mbps.mean(),
                'miss_rate_100ms': miss_rate_100ms, # Integrado en la tabla
                'stream': stream_id
            })
            
            temp_df = pd.DataFrame({
                'delay': delays,
                'stream': f"Stream {stream_id}",
                'gamma_crit': float(gamma_label)
            })
            raw_data_list.append(temp_df)
            
    return stats_rows, pd.concat(raw_data_list) if raw_data_list else pd.DataFrame()

# ==========================================
# BUCLE PRINCIPAL DE PROCESAMIENTO
# ==========================================
ROOT_DIR = "RESULTADOS_TESTBED_DSRC/results_SAT_MLWDF_RR_STBP/results/"
schednames = ["MLWDF", "RR", "STBP"]
# schednames = ["RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

for schedname in schednames:
    configs = subschedconfigs if schedname == "MLWDF" else [""]
    if schedname == "MLWDF":
        for config in configs:
            all_raw = []
            all_stats = []
            BASE_DIR = os.path.join(ROOT_DIR, schedname, config)
            if not os.path.exists(BASE_DIR): 
                continue
                
            gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
            if not gamma_folders: gamma_folders = [BASE_DIR]
            
            for folder in gamma_folders:
                match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
                g_val = match.group(1) if match else "1.0"
                
                s, r = get_full_data_and_stats(folder, g_val)
                all_stats.extend(s)
                all_raw.append(r)
                
            if all_stats:
                # 1. Exportación de CSV
                df_stats = pd.DataFrame(all_stats).sort_values(by=['stream', 'gamma'])
                
                # --- ACTUALIZADO: Añadidas columnas 'std' y 'miss_rate_100ms' a la exportación ---
                cols = ['gamma', 'lw', 'lq', 'med', 'uq', 'uw', 'xaux', 'avg', 'max', 'min', 'std', 'thput_mbps', 'miss_rate_100ms', 'stream']
                df_stats = df_stats[cols]
                
                config_suffix = f"_{config}" if config else ""
                csv_name = f"boxplot_{schedname}{config_suffix}.csv"
                df_stats.to_csv(csv_name, index=False, float_format='%.3f')
                
                print(f"\n{'-'*60}\nDATOS CONSOLIDADOS CON ÉXITO: {schedname} {config}")
                print(f"Archivo guardado: {csv_name}")
                print(f"{'-'*60}")
                
                # 2. Ordenamiento de Datos Crudos (con todos los delays "appended")
                df_final = pd.concat(all_raw).sort_values(by=['gamma_crit', 'stream']).reset_index(drop=True)

                # (Opcional) Llamadas a visualización comentadas:
                # plot_mean_sd(df_final, schedname, config)
                # plot_advanced_stats(df_final, schedname, config)
    else:
        print(f"\n{'-'*60}\nDATOS CONSOLIDADOS CON ÉXITO: {schedname} (sin subschedconfigs)")
        # Para schedulers que no son MLWDF, procesar sin subschedconfigs
        all_raw = []
        all_stats = []
        BASE_DIR = os.path.join(ROOT_DIR, schedname)
        if not os.path.exists(BASE_DIR): 
            continue
            
        s, r = get_full_data_and_stats(BASE_DIR, "No Sweep")
        all_stats.extend(s)
        all_raw.append(r)
        
        if all_stats:
            df_stats = pd.DataFrame(all_stats).sort_values(by=['stream', 'gamma'])
            cols = ['gamma', 'lw', 'lq', 'med', 'uq', 'uw', 'xaux', 'avg', 'max', 'min', 'std', 'thput_mbps', 'miss_rate_100ms', 'stream']
            df_stats = df_stats[cols]
            
            csv_name = f"boxplot_{schedname}.csv"
            df_stats.to_csv(csv_name, index=False, float_format='%.3f')
            
            print(f"\n{'-'*60}\nDATOS CONSOLIDADOS CON ÉXITO: {schedname}")
            print(f"Archivo guardado: {csv_name}")
            print(f"{'-'*60}")
            
            df_final = pd.concat(all_raw).sort_values(by=['gamma_crit', 'stream']).reset_index(drop=True)
    

## JUST VISUALIZING EVERYTHING IN ONE Fig.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from cycler import cycler

# ==========================================
# CONFIGURACIÓN GLOBAL DE ESTILO
# ==========================================
servicecolors = ["#115e67", "#f0b434", "#ff5733", "#800000"]
plt.rcParams['axes.prop_cycle'] = cycler(color=servicecolors)
sns.set_theme(style="whitegrid")

# ==========================================
# FUNCIONES DE APOYO (Simulación de dependencias)
# ==========================================

def get_full_data_and_stats(folder_path, gamma_label):
    """Calcula estadísticas y devuelve DataFrame long-form para Seaborn"""
    stats_rows = []
    raw_data_list = []
    
    for stream_id in ["1_1", "1_2"]:
        delays = process_stream_logs(folder_path, stream_id)
        
        if isinstance(delays, np.ndarray) and len(delays) > 0:
            # 1. Estadísticas para el CSV
            d_sorted = np.sort(delays)
            lq, med, uq = np.percentile(d_sorted, [25, 50, 75])
            iqr = uq - lq
            lw = d_sorted[d_sorted >= (lq - 1.5 * iqr)].min()
            uw = d_sorted[d_sorted <= (uq + 1.5 * iqr)].max()

            stats_rows.append({
                'gamma': float(gamma_label), 
                'lw': lw, 
                'lq': lq, 
                'med': med, 
                'uq': uq, 
                'uw': uw, 
                'xaux': iqr,
                'avg': d_sorted.mean(), 
                'max': d_sorted.max(),
                'min': d_sorted.min(),
                'stream': stream_id
            })
            
            # 2. Datos para Seaborn (Long-form)
            temp_df = pd.DataFrame({
                'delay': delays,
                'stream': f"Stream {stream_id}",
                'gamma_crit': float(gamma_label)
            })
            raw_data_list.append(temp_df)
            
    return stats_rows, pd.concat(raw_data_list) if raw_data_list else pd.DataFrame()

def plot_advanced_stats(df_plot, sched_name, config_name=""):
    """Genera PDF (KDE) y CDF (ECDF) con proyecciones analíticas de la media"""
    
    # Aseguramos orden numérico y de flujo estricto
    df_plot = df_plot.sort_values(by=['gamma_crit', 'stream']).reset_index(drop=True)
    unique_gammas = sorted(df_plot['gamma_crit'].unique())
    
    # 1. RENDERIZADO DE LA CDF CON DETECCIÓN DE MEDIAS
    # Usamos col_wrap=3 para distribuir los escenarios limpiamente en una cuadrícula
    g_cdf = sns.displot(
        data=df_plot, x="delay", hue="stream", col="gamma_crit",
        col_wrap=3, kind="ecdf", linewidth=2.5, height=4, aspect=1.2,
        palette=servicecolors[:2]
    )
    
    g_cdf.figure.suptitle(f"M-LWDF Convergence Analysis (CDF) - {sched_name} {config_name}", y=1.05, fontsize=14, fontweight='bold')

    for i, ax in enumerate(g_cdf.axes.flat):
        if i >= len(unique_gammas): 
            break
        
        current_gamma = unique_gammas[i]
        df_gamma = df_plot[df_plot['gamma_crit'] == current_gamma]
        
        ax.grid(True, which='both', linestyle=':', alpha=0.5)
        
        # Iterar sobre los flujos dentro de esta celda para pintar sus respectivas medias
        for idx, stream_id in enumerate(sorted(df_gamma['stream'].unique())):
            df_stream = df_gamma[df_gamma['stream'] == stream_id]
            delays = df_stream['delay'].values
            
            if len(delays) == 0: 
                continue
                
            # Calcular la media del flujo actual
            mean_val = np.mean(delays)
            # Calcular el percentil o altura empírica en la que corta la CDF
            f_at_mean = np.mean(delays <= mean_val)
            
            current_color = servicecolors[idx % len(servicecolors)]
            
            # Línea vertical discontinua para marcar la ubicación de la media
            ax.axvline(
                x=mean_val, 
                color=current_color, 
                linestyle='--', 
                linewidth=1.5, 
                alpha=0.7
            )
            
            # Punto de intersección sobre la curva
            ax.scatter(
                mean_val, 
                f_at_mean, 
                color=current_color, 
                s=40, 
                marker='o', 
                edgecolors='black', 
                zorder=5
            )
            
            # Línea horizontal fina para marcar la probabilidad teórica/empírica asociada
            ax.axhline(
                y=f_at_mean, 
                color=current_color, 
                linestyle=':', 
                linewidth=0.8, 
                alpha=0.4
            )
            
            ax.text(
                mean_val + (idx * 20), f_at_mean,
                f"Mean: {mean_val:.2f} ms\nCDF: {f_at_mean:.2f}",
                color=current_color)
            
        # Acotar los ejes por subgráfico para evitar desbordamientos por colas largas
        max_visible_delay = df_gamma['delay'].quantile(0.98)
        ax.set_xlim(0, max_visible_delay * 1.05 if max_visible_delay > 0 else 100)
        ax.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.show()

# ==========================================
# BUCLE PRINCIPAL DE PROCESAMIENTO
# ==========================================
# NOTA: Cambia esta ruta ficticia por la ubicación real de tu carpeta de resultados de ns-3
ROOT_DIR =  "./RESULTADOS_TESTBED_DSRC/results_SAT_MLWDF_RR_STBP/results"
schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

# Simulamos la creación de carpetas para que no falle si corres el script aislado:
for s in schednames:
    conf_list = subschedconfigs if s == "MLWDF" else [""]
    for c in conf_list:
        path = os.path.join(ROOT_DIR, s, c, "GAMMA_CRIT_1.0")
        os.makedirs(path, exist_ok=True)

for schedname in schednames:
    configs = subschedconfigs if schedname == "MLWDF" else [""]
    
    for config in configs:
        all_raw = []
        all_stats = []
        BASE_DIR = os.path.join(ROOT_DIR, schedname, config)
        if not os.path.exists(BASE_DIR): 
            continue
            
        gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
        if not gamma_folders: 
            gamma_folders = [BASE_DIR]
        
        for folder in gamma_folders:
            match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
            g_val = match.group(1) if match else "1.0" # Valor por defecto seguro
            
            s, r = get_full_data_and_stats(folder, g_val)
            
            # GUARDADO SEGURO: Eliminamos la asignación inplace defectuosa que causaba desajustes
            all_stats.extend(s)
            all_raw.append(r)
            
        if all_stats:
            # --- 1. Exportación de CSV ordenado numéricamente ---
            df_stats = pd.DataFrame(all_stats).sort_values(by=['stream', 'gamma'])
            cols = ['gamma', 'lw', 'lq', 'med', 'uq', 'uw', 'xaux', 'avg', 'max', 'min', 'stream']
            df_stats = df_stats[cols]
            
            config_suffix = f"_{config}" if config else ""
            csv_name = f"boxplot_{schedname}{config_suffix}.csv"
            df_stats.to_csv(csv_name, index=False, float_format='%.3f')
            
            print(f"\n{'-'*50}")
            print(f"DATOS CONSOLIDADOS CON ÉXITO: {schedname} {config}")
            print(f"{'-'*50}")
            print(df_stats.to_string(index=False))

            # --- 2. Preparación y Ordenamiento estricto de Datos Crudos ---
            df_final = pd.concat(all_raw)
            # Solución definitiva: Ordenamos globalmente antes de renderizar
            df_final = df_final.sort_values(by=['gamma_crit', 'stream']).reset_index(drop=True)
            gamma_order = sorted(df_final['gamma_crit'].unique())

            # --- 3. Visualización: Boxplot ---
            plt.figure(figsize=(10, 6))
            sns.boxplot(
                data=df_final,
                x='gamma_crit',
                y='delay',
                hue='stream',
                order=gamma_order,
                showfliers=True,
                showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 7},
                palette=servicecolors[:2]
            )
            
            plt.title(f"Delay Distribution Profile: {schedname} {config}", fontsize=12, fontweight='bold')
            plt.ylabel("Retardo (ms)")
            plt.xlabel(r"Gamma Crítico ($\gamma_{crit}$)")
            plt.grid(True, linestyle=':', alpha=0.6, axis='y')
            plt.tight_layout()
            plt.show()
            
            # --- 4. Visualización Avanzada: Curvas CDF + Proyecciones de la Media ---
            plot_advanced_stats(df_final, schedname, config)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from cycler import cycler

# ==========================================
# CONFIGURACIÓN GLOBAL DE ESTILO
# ==========================================
servicecolors = ["#115e67", "#f0b434", "#ff5733", "#800000"]
plt.rcParams['axes.prop_cycle'] = cycler(color=servicecolors)
sns.set_theme(style="whitegrid")

# ==========================================
# FUNCIONES DE APOYO
# ==========================================

def get_full_data_and_stats(folder_path, gamma_label):
    """Calcula estadísticas y devuelve DataFrame long-form para Seaborn"""
    stats_rows = []
    raw_data_list = []
    
    for stream_id in ["1_1", "1_2"]:
        # Se asume que process_stream_logs está definida y devuelve np.array
        delays = process_stream_logs(folder_path, stream_id)
        
        if isinstance(delays, np.ndarray) and len(delays) > 0:
            # 1. Estadísticas para el CSV (Formato exacto pedido)
            d_sorted = np.sort(delays)
            lq, med, uq = np.percentile(d_sorted, [25, 50, 75])
            iqr = uq - lq
            lw = d_sorted[d_sorted >= (lq - 1.5 * iqr)].min()
            uw = d_sorted[d_sorted <= (uq + 1.5 * iqr)].max()

            stats_rows.append({
                'gamma': float(gamma_label), 
                'lw': lw, 
                'lq': lq, 
                'med': med, 
                'uq': uq, 
                'uw': uw, 
                'xaux': iqr,
                'avg': d_sorted.mean(), 
                'max': d_sorted.max(),
                'min': d_sorted.min(),
                'stream': stream_id # Nombre corto: 1_1 o 1_2
            })
            
            # 2. Datos para Seaborn (Long-form)
            temp_df = pd.DataFrame({
                'delay': delays,
                'stream': f"Stream {stream_id}",
                'gamma_crit': float(gamma_label)
            })
            raw_data_list.append(temp_df)
            
    return stats_rows, pd.concat(raw_data_list) if raw_data_list else pd.DataFrame()

def plot_advanced_stats(df_plot, sched_name, config_name=""):
    """Genera PDF (KDE) y CDF (ECDF) con la paleta de colores personalizada"""
    df_plot = df_plot.sort_values(by=['gamma_crit', 'stream'])
    
    # 1. KDE (PDF)
    g_pdf = sns.displot(
        data=df_plot, x="delay", hue="stream", col="gamma_crit",
        col_wrap=3, kind="kde", fill=True, common_norm=False,
        height=4, aspect=1.2, alpha=0.3, palette=servicecolors[:2]
    )
    g_pdf.figure.suptitle(f"PDF - {sched_name} {config_name}", y=1.02, fontsize=16)
    
    # 2. ECDF (CDF)
    percentile_target = 0.90
    g_cdf = sns.displot(
        data=df_plot, x="delay", hue="stream", col="gamma_crit",
        col_wrap=3, kind="ecdf", linewidth=2, height=4, aspect=1.2,
        palette=servicecolors[:2]
    )
    
    stats_p90 = df_plot.groupby(['gamma_crit', 'stream'])['delay'].quantile(percentile_target).reset_index()
    unique_gammas = sorted(df_plot['gamma_crit'].unique())

    for i, ax in enumerate(g_cdf.axes.flat):
        if i >= len(unique_gammas): break
        current_gamma = unique_gammas[i]
        ax.grid(True, which='both', linestyle=':', alpha=0.6)
        ax.axhline(percentile_target, color='#800000', linestyle='--', alpha=0.5)
        
        subset_stats = stats_p90[stats_p90['gamma_crit'] == current_gamma]
        for idx, row in subset_stats.iterrows():
            # Usar colores de la lista para las líneas verticales
            color_txt = servicecolors[0] if "1_1" in row['stream'] else servicecolors[1]
            ax.axvline(row['delay'], ymax=percentile_target, color=color_txt, linestyle=':', alpha=0.8)
    
    plt.show()

# ==========================================
# BUCLE PRINCIPAL DE PROCESAMIENTO
# ==========================================
ROOT_DIR = "RESULTADOS_TESTBED_DSRC/results_MLWDF_STBP_RR/results/"
schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

for schedname in schednames:
    configs = subschedconfigs if schedname == "MLWDF" else [""]
    
    for config in configs:
        all_raw = []
        all_stats = []
        BASE_DIR = os.path.join(ROOT_DIR, schedname, config)
        if not os.path.exists(BASE_DIR): continue
            
        gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
        if not gamma_folders: gamma_folders = [BASE_DIR]
        
        for folder in gamma_folders:
            match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
            g_val = match.group(1) if match else "0.0"
            s, r = get_full_data_and_stats(folder, g_val)
            df_save = r.sort_values(by=['gamma_crit', 'stream'], inplace=True)
            
            
            all_stats.extend(s)
            all_raw.append(r)
            
        if all_stats:
            # --- 1. Exportación de CSV ordenado por Stream y luego Gamma ---
            df_stats = pd.DataFrame(all_stats).sort_values(by=['stream', 'gamma'])
            cols = ['gamma', 'lw', 'lq', 'med', 'uq', 'uw', 'xaux', 'avg', 'max', 'min', 'stream']
            df_stats = df_stats[cols]
            
            # Formato de nombre de archivo consistente
            config_suffix = f"_{config}" if config else ""
            csv_name = f"boxplot_{schedname}{config_suffix}.csv"
            df_stats.to_csv(csv_name, index=False, float_format='%.3f')
            
            print(f"\n{'-'*30}")
            print(f"DATOS CONSOLIDADOS: {schedname} {config}")
            print(f"{'-'*30}")
            print(df_stats.to_string(index=False))

            # --- 2. Visualización: Boxplot ---
            df_final = pd.concat(all_raw)
            plt.figure(figsize=(10, 6))
            gamma_order = sorted(df_final['gamma_crit'].unique())

            sns.boxplot(
                data=df_final,
                x='gamma_crit',
                y='delay',
                hue='stream',
                order=gamma_order,
                showfliers=False,
                showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 8},
                palette=servicecolors[:2]
            )
            
            plt.title(f"Distribución de Retardos: {schedname} {config}")
            plt.ylabel("Retardo (ms)")
            plt.xlabel("Gamma Crítico")
            plt.show()
            
            # # --- 3. Visualización: PDF y CDF ---
            # plot_advanced_stats(df_final, schedname, config)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re



# ==========================================
# PROCESAMIENTO Y RECOLECCIÓN
# ==========================================
all_data_frames = []

for schedname in schednames:
    if schedname == "MLWDF":
        for config in subschedconfigs:
            BASE_DIR = os.path.join(rOOT_DIR, schedname, config)
            gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
            
            for folder in gamma_folders:
                if not os.path.isdir(folder): continue
                # Extraer Gamma
                match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
                g_val = f"{float(match.group(1)):.1f}" if match else "0.0"
                
                # Procesar streams
                for stream_id in ["1_1", "1_2"]:
                    delays = process_stream_logs(folder, stream_id)
                    if isinstance(delays, np.ndarray) and len(delays) > 0:
                        all_data_frames.append(pd.DataFrame({
                            'Delay (ms)': delays,
                            'Stream': stream_id,
                            'Config': config,
                            'Scheduler': schedname,
                            'X_Axis': f"{schedname}\n{config}\nΓ={g_val}"
                        }))
    else:
        # Caso RR y STBP (No sweep)
        BASE_DIR = os.path.join(rOOT_DIR, schedname)
        if os.path.isdir(BASE_DIR):
            for stream_id in ["1_1", "1_2"]:
                delays = process_stream_logs(BASE_DIR, stream_id)
                if isinstance(delays, np.ndarray) and len(delays) > 0:
                    all_data_frames.append(pd.DataFrame({
                        'Delay (ms)': delays,
                        'Stream': stream_id,
                        'Config': 'N/A',
                        'Scheduler': schedname,
                        'X_Axis': f"{schedname}\n(No sweep)"
                    }))

# Consolidar todos los datos en un solo DataFrame
df_master = pd.concat(all_data_frames, ignore_index=True)

# ==========================================
# PINTAR EN LA MISMA FIGURA
# ==========================================
plt.figure(figsize=(16, 8))

# El boxplot comparará todas las combinaciones en el eje X
ax = sns.boxplot(
    data=df_master, 
    x='X_Axis', 
    y='Delay (ms)', 
    hue='Stream', 
    palette="muted",
    showfliers=False  # Ocultamos outliers para que se vean bien las cajas
)

plt.title("Comparativa Global de Retardos: RR y STBP vs MLWDF (Barrido Gamma)", fontsize=16)
plt.xlabel("Planificador / Configuración / Gamma", fontsize=12)
plt.ylabel("Retardo (ms)", fontsize=12)
plt.xticks(rotation=45) # Rotamos etiquetas para que no se solapen
plt.legend(title="Flujo", loc='upper right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Configuración de estilo
sns.set_theme(style="whitegrid")

# ==========================================
# FUNCIONES DE APOYO
# ==========================================

def get_full_data_and_stats(folder_path, gamma_label):
    """Calcula estadísticas y devuelve DataFrame long-form para Seaborn"""
    stats_rows = []
    raw_data_list = []
    
    for stream_id in ["1_1", "1_2"]:
        # Asumiendo que process_stream_logs está definida en tu entorno
        delays = process_stream_logs(folder_path, stream_id)
        
        if isinstance(delays, np.ndarray) and len(delays) > 0:
            # 1. Estadísticas para CSV
            d_sorted = np.sort(delays)
            lq, med, uq = np.percentile(d_sorted, [25, 50, 75])
            iqr = uq - lq
            lw = d_sorted[d_sorted >= (lq - 1.5 * iqr)].min()
            uw = d_sorted[d_sorted <= (uq + 1.5 * iqr)].max()

            stats_rows.append({
                'gamma': float(gamma_label), 'lw': lw, 'lq': lq, 'med': med, 
                'uq': uq, 'uw': uw, 'avg': d_sorted.mean(), 'stream': f"Stream {stream_id}"
            })
            
            # 2. Datos para Seaborn
            temp_df = pd.DataFrame({
                'delay': delays,
                'stream': f"Stream {stream_id}",
                'gamma_crit': float(gamma_label)
            })
            raw_data_list.append(temp_df)
            
    return stats_rows, pd.concat(raw_data_list) if raw_data_list else pd.DataFrame()

def plot_advanced_stats(df_plot, sched_name, config_name=""):
    """Genera PDF (KDE) y CDF (ECDF) a partir del DataFrame procesado"""
    
    # Asegurar orden numérico para las gráficas
    df_plot = df_plot.sort_values(by=['gamma_crit', 'stream'])
    
    # --- 1. KDE (PDF) ---
    g_pdf = sns.displot(
        data=df_plot, x="delay", hue="stream", col="gamma_crit",
        col_wrap=3, kind="kde", fill=True, common_norm=False,
        height=4, aspect=1.2, alpha=0.3
    )
    g_pdf.figure.suptitle(f"PDF - {sched_name} {config_name}", y=1.02, fontsize=16)
    plt.show()

    # --- 2. ECDF (CDF) ---
    percentile_target = 0.90 # Ajustado a 0.90 (90%) como en tu lógica de texto
    g_cdf = sns.displot(
        data=df_plot, x="delay", hue="stream", col="gamma_crit",
        col_wrap=3, kind="ecdf", linewidth=2, height=4, aspect=1.2
    )
    
    stats_p90 = df_plot.groupby(['gamma_crit', 'stream'])['delay'].quantile(percentile_target).reset_index()
    unique_gammas = sorted(df_plot['gamma_crit'].unique())

    for i, ax in enumerate(g_cdf.axes.flat):
        if i >= len(unique_gammas): break
        
        current_gamma = unique_gammas[i]
        ax.grid(True, which='both', linestyle=':', alpha=0.6)
        ax.axhline(percentile_target, color='red', linestyle='--', alpha=0.5)

        subset_stats = stats_p90[stats_p90['gamma_crit'] == current_gamma]
        for _, row in subset_stats.iterrows():
            color_txt = "C0" if "1_1" in row['stream'] else "C1"
            offset_y = 0.85 if "1_1" in row['stream'] else 0.50
            
            ax.axvline(row['delay'], ymax=percentile_target, color=color_txt, linestyle=':', alpha=0.6)
            ax.text(row['delay'], offset_y, f"{row['delay']:.1f} ms", color=color_txt,
                    rotation=90, fontweight='bold', fontsize=9, 
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    g_cdf.figure.suptitle(f"CDF - {sched_name} {config_name}", y=1.02, fontsize=16)
    plt.show()

# ==========================================
# BUCLE PRINCIPAL DE PROCESAMIENTO
# ==========================================
ROOT_DIR = "RESULTADOS_TESTBED_DSRC/results_MLWDF_STBP_RR/results/"
schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

for schedname in schednames:
    if schedname == "MLWDF":
        for config in subschedconfigs:
            all_raw = []
            BASE_DIR = os.path.join(ROOT_DIR, schedname, config)
            gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
            
            for folder in gamma_folders:
                match = re.search(r"GAMMA_CRIT_([\d\.]+)", folder)
                if match:
                    g_val = match.group(1)
                    _, r = get_full_data_and_stats(folder, g_val)
                    all_raw.append(r)
            
            if all_raw:
                df_final = pd.concat(all_raw)
                # 1. Boxplot Original
                plt.figure(figsize=(10, 6))
                sns.boxplot(data=df_final, x='gamma_crit', y='delay', hue='stream', showfliers=False)
                plt.title(f"Distribución: {schedname} ({config})")
                plt.show()
                
                # 2. PDF y CDF Integradas
                plot_advanced_stats(df_final, schedname, config)
    else:
        # Lógica para RR y STBP (simplificada)
        BASE_DIR = os.path.join(ROOT_DIR, schedname)
        if os.path.isdir(BASE_DIR):
            _, df_final = get_full_data_and_stats(BASE_DIR, 0.0) # 0.0 como placeholder
            plot_advanced_stats(df_final, schedname)

In [ ]:
# ==========================================
# ANÁLISIS TEMPORAL: SOJOURN TIME (Integrado)
# ==========================================
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT_DIR = "RESULTADOS_TESTBED_DSRC/results_SAT_MLWDF_RR_STBP/results/"
schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]
nstreams = 2

for schedname in schednames:
    # --- Definición de las rutas a procesar ---
    if schedname == "MLWDF":
        # Para MLWDF iteramos por cada configuración (W1_Q1, etc.)
        configs_to_process = subschedconfigs
    else:
        # Para RR y STBP solo hay una "configuración" (la raíz del scheduler)
        configs_to_process = [""] 

    for config in configs_to_process:
        BASE_DIR = os.path.join(ROOT_DIR, schedname, config)
        if not os.path.exists(BASE_DIR):
            continue

        # Buscamos las carpetas de Gamma
        gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
        
        # Si no hay carpetas GAMMA_CRIT (caso RR/STBP sin sweep), procesamos la BASE_DIR
        if not gamma_folders:
            gamma_folders = [BASE_DIR]

        # Crear subplots: uno por cada carpeta de Gamma encontrada
        fig, axes = plt.subplots(1, len(gamma_folders), 
                                 figsize=(5 * len(gamma_folders), 5), 
                                 sharey=True)
        
        # Asegurar que axes sea siempre una lista (para casos de 1 solo plot)
        if len(gamma_folders) == 1:
            axes = [axes]

        for idx, folder in enumerate(gamma_folders): 
            # Extraer valor de Gamma para el título
            try:
                if "GAMMA_CRIT_" in folder:
                    gamma_val = folder.split('_')[-1]
                else:
                    gamma_val = "N/A"
            except:
                gamma_val = "Unknown"
                
            print(f"Procesando Temporal: {schedname} {config} | Gamma: {gamma_val}")

            # Buscar el archivo de log
            stb_logs_pattern = os.path.join(folder, "**", "go_client", f"logSchedulerSojournTime_{schedname}.log")
            stb_found_list = glob.glob(stb_logs_pattern, recursive=True)
        
            if not stb_found_list:
                print(f"  -> No se encontró log en {folder}")
                continue

            logfile_path = stb_found_list[0]
            
            try:
                read_buff = pd.read_csv(logfile_path, delimiter="\t", 
                                        names=["Time", "Str", "STB", "maxdel"])
            except Exception as e:
                print(f"  -> Error leyendo archivo: {e}")
                continue

            if not read_buff.empty:
                start_time = read_buff["Time"].iloc[0]
                
                for n in range(nstreams):
                    # Filtro por ID de Stream (0, 4, 8...)
                    read_buff_filtered = read_buff[read_buff["Str"] == n*4]
                    
                    if not read_buff_filtered.empty:
                        # Conversión: microsegundos -> segundos (1e-6)
                        x_data = (read_buff_filtered["Time"] - start_time) * 1e-6

                        # Sojourn time en segundos
                        y_data = read_buff_filtered["STB"] * 1e-6

                        # --------- Stability metric: running mean ----------
                        y_cummean = np.cumsum(y_data) / np.arange(1, len(y_data) + 1) 
                        
                        
                        # Plot con submuestreo para rendimiento
                        axes[idx].plot(x_data[::500], y_cummean[::500], 
                                       linestyle='--', label=f"Stream {n+1} (Mean Rate)")
                        # axes[idx].plot(x_data[::500], y_data[::500], 
                        #                linestyle='--', label=f"Stream {n+1} (STB)")

            # Estética de la sub-gráfica
            title_suffix = f"({config})" if config else ""
            axes[idx].set_title(f"{schedname} {title_suffix}\n$\gamma$={gamma_val}")
            axes[idx].set_xlabel("Time (s)")
            axes[idx].grid(True, linestyle=':', alpha=0.6)

            if idx == 0:
                axes[idx].set_ylabel("Sojourn Time (s)")
            
            axes[idx].legend()

        plt.tight_layout()
        plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


stb_logs_pattern = os.path.join(folder, "*.log")
stb_logs_pattern = os.path.join(folder, "**", "LOGS_congestionWindow", "*.log")

# Buscar archivos
stb_found_list = glob.glob(stb_logs_pattern)
print(stb_found_list)
if not stb_found_list:
    print(f"  -> No se encontró ningún log en {folder}")
else:
    # Ordenar para consistencia (opcional)
    stb_found_list.sort()
    
    # Inicializar variables
    lost_files = None
    cwnd_file = None
    for f in stb_found_list:
        base_name = os.path.basename(f)
        if base_name.startswith("OnPacketLost"):
            lost_files = f
            # print(f)
        elif base_name.startswith("maybeIncreaseCwnd"):
            cwnd_file = f
            # print(f)
        elif base_name.startswith("logchannel"):
            logchannel_file = f
            # print(f)
    # Comprobación de que se encontraron ambos
    if lost_files is None:
        print("  -> No se encontró el log de pérdidas (OnPacketLost).")
    if cwnd_file is None:
        print("  -> No se encontró el log de congestion window (CWND).")

    
    df_lost = pd.read_csv(lost_files, sep='\s+', usecols=[0], names=["Time"], engine='python')
    
    # 2. Leer LOG DE CWND
    # Asumimos que maybeIncreaseCwnd.log tiene: Time, Bytes
    df_cwnd = pd.read_csv(cwnd_file, sep='\s+', names=["Time", "CwndBytes"], engine='python')

    # Convertir a numérico por seguridad
    df_lost["Time"] = pd.to_numeric(df_lost["Time"], errors='coerce')
    df_cwnd["Time"] = pd.to_numeric(df_cwnd["Time"], errors='coerce')
    df_cwnd["CwndBytes"] = pd.to_numeric(df_cwnd["CwndBytes"], errors='coerce')



# 3. SINCRONIZACIÓN (La mejora clave)
# Ordenamos por tiempo (necesario para merge_asof)
df_lost = df_lost.sort_values("Time")
df_cwnd = df_cwnd.sort_values("Time")
inter_arrival_time = np.mean(np.diff(df_cwnd["Time"]))
# MERGE_ASOF: Para cada evento de pérdida, busca el valor de CWND 
# registrado en el instante anterior más cercano ('backward').
# Esto nos da la altura exacta de la X roja.
df_lost_merged = pd.merge_asof(df_lost, df_cwnd, on="Time", direction="backward")

# 4. ESTADÍSTICAS
total_loss_events = len(df_lost)
total_cwnd_updates = len(df_cwnd)
total_events = total_loss_events + total_cwnd_updates

loss_ratio = (total_loss_events / total_events * 100) if total_events > 0 else 0

print(f"--- RESULTADOS ---")
print(f"Eventos de pérdida (Loss): {total_loss_events}")
print(f"Actualizaciones de CWND (Success): {total_cwnd_updates}")
print(f"Ratio de Pérdida (aprox): {loss_ratio:.2f}%")
print(f"Promedio de CWND: {df_cwnd['CwndBytes'].mean():.0f} bytes")
print(f"Promedio ventana actualización: {inter_arrival_time*1e-3:.2f} ms")
print(f"Capacidad del canal promedio (aprox): {(df_cwnd['CwndBytes'].mean()/inter_arrival_time):.2f} Mbps")

plt.figure(figsize=(12, 6))

# Normalizar tiempo para que empiece en 0 (opcional, ayuda a leer mejor)
start_time = df_cwnd["Time"].min()
t_cwnd = df_cwnd["Time"] - start_time
t_lost = df_lost_merged["Time"] - start_time



plt.plot(t_cwnd*1e-6, df_cwnd["CwndBytes"], label="CWND (CUBIC)", color="#0056b3", linewidth=1.2)

y_vals_loss = [df_cwnd["CwndBytes"].mean()] * len(df_lost) # Los pintamos a altura media para verlos
plt.scatter(t_lost*1e-6, df_lost_merged["CwndBytes"], color="red", marker="x", label="Packet Loss")

plt.title(f"Análisis Congestión CUBIC (Loss Ratio Eventos: {loss_ratio:.2f}%)")
plt.xlabel("Tiempo de ejecución (s)")
plt.ylabel("Ventana de Congestión (Bytes)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
# plt.xlim(0,3)
plt.show()




df = pd.read_csv(
    logchannel_file,
    sep=r"\s+",
    header=None,
    names=["ts_us", "min_rtt", "latest_rtt", "srtt", "cwnd"]
)

# Eliminar el sufijo 'ms' y convertir a float
for col in ["min_rtt", "latest_rtt", "srtt"]:
    df[col] = df[col].str.replace("ms", "", regex=False).astype(float)

# Convertir RTT a segundos
df["srtt_s"] = df["srtt"] * 1e-3

# Convertir timestamp a segundos relativos
df["t_s"] = (df["ts_us"] - df["ts_us"].iloc[0]) * 1e-6

# Calcular pacing rate
df["pacing_rate_Bps"] = df["cwnd"] / df["srtt_s"]
df["pacing_rate_Mbps"] = df["pacing_rate_Bps"] * 8 / 1e6

# Valor medio
mean_rate = df["pacing_rate_Mbps"].mean()

print(f"Pacing rate medio: {mean_rate:.2f} Mbps")

# -------- PLOT --------
plt.figure()
plt.plot(df["t_s"], df["pacing_rate_Mbps"])
plt.xlabel("Time [s]")
plt.ylabel("Pacing rate [Mbps]")
plt.title("QUIC pacing rate = cwnd / SRTT")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================================
# ANÁLISIS DE ESTABILIDAD: PACING RATE (THROUGHPUT ESTIMADO)
# ==========================================================
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Reutilizamos tu función de conversión
def parse_rtt_to_seconds(x):
    if isinstance(x, str):
        x = x.strip()
        if x.endswith("ms"): return float(x.replace("ms", "")) * 1e-3
        if x.endswith("s"): return float(x.replace("s", ""))
    return float(x)

ROOT_DIR = "RESULTADOS_TESTBED_DSRC/results_SAT_MLWDF_RR_STBP/results/"
schednames = ["MLWDF", "RR", "STBP"]
subschedconfigs = ["W1_Q1", "W1_Q0", "W0_Q1"]

for schedname in schednames:
    # Definir carpetas a procesar según el scheduler
    configs_to_process = subschedconfigs if schedname == "MLWDF" else [""] 

    for config in configs_to_process:
        BASE_DIR = os.path.join(ROOT_DIR, schedname, config)
        if not os.path.exists(BASE_DIR): continue

        gamma_folders = sorted(glob.glob(os.path.join(BASE_DIR, "GAMMA_CRIT_*")))
        if not gamma_folders: gamma_folders = [BASE_DIR]

        # Crear figura
        fig, axes = plt.subplots(1, len(gamma_folders), figsize=(5 * len(gamma_folders), 5), sharey=True)
        if len(gamma_folders) == 1: axes = [axes]

        for idx, folder in enumerate(gamma_folders):
            gamma_val = folder.split("_")[-1] if "GAMMA_CRIT_" in folder else "N/A"
            
            # Localizar el log de congestión/channel
            log_pattern = os.path.join(folder, "**", "LOGS_congestionWindow", "logchannel.log")
            found_logs = glob.glob(log_pattern, recursive=True)

            if not found_logs:
                print(f"  -> No se encontró log de congestion en {folder}")
                continue

            # Leer el log (asumiendo el formato ts_us, min_rtt, latest_rtt, srtt, cwnd)
            try:
                df = pd.read_csv(
                    found_logs[0],
                    sep=r"\s+",
                    header=None,
                    names=["ts_us", "min_rtt", "latest_rtt", "srtt", "cwnd"]
                )
            except Exception as e:
                print(f"  -> Error leyendo {found_logs[0]}: {e}")
                continue

            if not df.empty:
                # 1. Limpieza y conversión de RTT
                df["srtt_s"] = df["srtt"].apply(parse_rtt_to_seconds)
                df["t_s"] = (df["ts_us"] - df["ts_us"].iloc[0]) * 1e-6

                # 2. Cálculo del Pacing Rate (bps = cwnd_bytes / srtt_seconds)
                # Multiplicamos por 8 para bits y dividimos por 1e6 para Mbps
                df["pacing_rate_Mbps"] = (df["cwnd"] / df["srtt_s"]) * 8 / 1e6
                
                # Filtrar posibles infinitos si srtt fue 0
                df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["pacing_rate_Mbps"])

                mean_rate = df["pacing_rate_Mbps"].mean()
                
                # 3. Graficar
                # Usamos submuestreo [::10] si el log es muy denso para agilizar el plot
                axes[idx].plot(df["t_s"][::10], df["pacing_rate_Mbps"][::10], 
                               label=f"Avg: {mean_rate:.2f} Mbps", alpha=0.8)

                # Estética
                title_suffix = f"({config})" if config else ""
                axes[idx].set_title(f"{schedname} {title_suffix}\n$\gamma$={gamma_val}")
                axes[idx].set_xlabel("Time (s)")
                axes[idx].grid(True, linestyle=":", alpha=0.6)
                axes[idx].legend(loc="upper right", fontsize="small")

            if idx == 0:
                axes[idx].set_ylabel("Pacing Rate [Mbps]")

        plt.tight_layout()
        plt.show()